<a href="https://colab.research.google.com/github/nabilah-afrin/recommendation_system_rokomri_books/blob/secondary/notebooks/preprocessing_bn_books_title.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# %cd /content/drive/MyDrive/Rokomari Recommendation Dataset

%cd /content/drive/MyDrive/Dataset/rokomari_books/Rokomari Recommendation Dataset/Datasets

/content/drive/.shortcut-targets-by-id/1SdeIcOv6xY-c8y2UcMwPLYx_BaB0zdXx/Rokomari Recommendation Dataset/Datasets


In [3]:
!ls

 corrected_language.csv       rokomari_book_data_v2.csv		  rokomari_v2.ipynb
'Data Analysis Report.gdoc'   rokomari_books_only_bangla_v2.csv  'scraping log.txt'
 rokomari_book_data.csv       rokomari_v2.csv			  wrong_language_url.txt


In [7]:
# !pip install googletrans==4.0.0-rc1
# !pip install googletrans==4.0.0rc1 --quiet

In [5]:
# !pip install langdetect --quiet

In [8]:
!pip install deep_translator --quiet

# Import

In [9]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import googletrans as gt
import deep_translator as dt
from deep_translator import GoogleTranslator

# Useful fns

In [20]:
import re
import time
# from easynmt import EasyNMT
# model = EasyNMT('m2m_100_1.2B')
# print(model.translate('Bank Job Entrance Exam Preparation', target_lang='bn'))

def detect_language(row):
    try:
        lang = ld.detect(str(row))
    except:
        return 'unknown'
    return lang

def keep_only_bangla(row):
    """
    The unicode range between \u0980 - \u09FF defines the Bangla characters
    and digits in the Unicode character set
    """
    # return re.sub(r'[^\u0980-\u09FF ]+', '', str(row)).strip()
    return re.sub(r'[^\u0980-\u09FF\u0041-\u005A\u0061-\u007A :?]+', '', str(row)).strip()

def translate_author_names(row):
    text = str(row)
    # time.sleep(8)
    translator = gt.Translator()
    translation = translator.translate(text=text, dest='bn')

    return translation.text

def translate_to_bangla(row):
    return GoogleTranslator(source='auto', target='bn').translate(text=str(row))



# 1. Load Dataframe

In [13]:
df_bn = pd.read_csv("rokomari_books_only_bangla_v2.csv")

In [14]:
df_bn.head()

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,rating,n_ratings,n_reviews,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating
0,350167,অমানুষিক,মনোরঞ্জন ব্যাপারী,একা,Eka (India),পশ্চিমবঙ্গের বই,West Bengal Books,6 March 2023,9789357762298,No summary,...,5.0,0,0,450.0,450.0,https://www.rokomari.com/book/350167/amanushik,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0
1,377700,নির্বাচিত গল্প সংকলন,লু স্যুন,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: সমকালীন গল্প,West Bengal Books: Contemporary Story,Edition,9788194097303,No summary,...,5.0,0,0,320.0,320.0,https://www.rokomari.com/book/377700/nirbachit...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0
2,377718,তিমুর ও তার দলবল,আর্কাদি গাইদার,ছাড়পত্র প্রকাশন (ইন্ডিয়া),Charpatra Prakashan (India),পশ্চিমবঙ্গের বই: শিশু-কিশোর উপন্যাস,West Bengal Books: Children and Teens Novel,Edition,9788193860939,No summary,...,5.0,0,0,180.0,180.0,https://www.rokomari.com/book/377718/timur-o-t...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0
3,189529,চেক ডিসঅনার মামলার সহজ ভাষ্য,মোঃ কাইছার হামিদ,এ.কে লিগ্যাল সল্যুশন,A.K Legal Solution,ব্যাংকিং এন্ড কমার্স ল,Banking and Commerce Law,1st Published,No ISBN,* চেক ডিসঅনার ও মামলা দায়েরের পদ্ধতি সংক্রান্...,...,5.0,16,14,250.0,175.0,https://www.rokomari.com/book/189529/cheque-di...,https://img.cf.rokomari.com/ProductNew20190903...,Not Available,book,5.0
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,Advocacy/ Adjudication Law,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,5.0,0,0,750.0,500.0,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0


In [15]:
df_bn['language'].value_counts()

,count
language,
বাংলা,200867
Bangla & Arabic,2462
Bangla & English,1776
"Bangla, English, Arabic",134
"Bangla, Arabic, Urdu",130
"Arbi, Urdu, Bangla, English",12
Bangla & Korean,8


In [18]:
df_bn.loc[df_bn['language'] == 'Bangla & English']

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,rating,n_ratings,n_reviews,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating
4,325781,হাইকোর্ট এক্সাম ফর্মুলা,মোঃ কাইছার হামিদ,A.K Legal Solution,A.K Legal Solution,Advocacy/ Adjudication Law,Advocacy/ Adjudication Law,Edition,9789843545589,"""High Court Exam Formula"" book will be very he...",...,5.0,0,0,750.0,500.0,https://www.rokomari.com/book/325781/high-cour...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0
7,313765,ফিরে দেখা ১৯৭১ এর চিঠি ও কথা,ডঃ এ.কে.এম.এ. কাদের,A-Z Foundation,A-Z Foundation,"Diary, Letters and Memories","Diary, Letters and Memories",1st Edition,9789843542939,"The letters, clippings, scribbles, and documen...",...,5.0,2,1,850.0,731.0,https://www.rokomari.com/book/313765/reminisce...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,5.0
23,113881,ঈশান ও তার সুপারহিরো বন্ধুরা,এ. এন. এম নূরুল হক,Adorn Books For Children (ABC),Adorn Books For Children (ABC),Story: Children and Teens,Story: Children and Teens,1st Published,9789842010637,No summary,...,5.0,2,0,100.0,86.0,https://www.rokomari.com/book/113881/eshan-and...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,5.0
271,49102,ইসলামের ইতিবৃত্ত,তাজুল ইসলাম,Amena Prakashani,Amena Prakashani,Islamic History and Tradition,Islamic History and Tradition,2nd Published,9789843343543,। অধ্যায় ১ : বিশ্বজগৎ সৃষ্টির প্রথম পর্যায়। অধ...,...,5.0,0,0,450.0,360.0,https://www.rokomari.com/book/49102/a-brief-hi...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0
279,231495,"অ্যাম্বিশন এ প্লাস বুলেটিন সৃজনশীল সাজেশন, এসএ...",রাশেদুল কবির খোকন,Ambition Publications,Ambition Publications,HSC Suggestion: Arts Department,HSC Suggestion: Arts Department,Edition,No ISBN,AMBITION A বুলেটিন সৃজনশীল সাজেশনbr ২০২২ সালের...,...,4.0,3,1,185.0,120.0,https://www.rokomari.com/book/231495/ambition-...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204979,432557,"এডমিশন পকেট বুক, ইংলিশ গ্রামার ও লিটারেচার",মোঃ রোকনুজ্জামান সোহেল,Young Bengal Publications,Young Bengal Publications,University Admission Preparation,University Admission Preparation,1st Published,No ISBN,"ঢাবি, রাবি, চবি, জাবি, গুচ্ছ , Ari ( Cluster) ...",...,5.0,0,0,180.0,180.0,https://www.rokomari.com/book/432557/admission...,https://img.cf.rokomari.com/ProductNew20190903...,Request For Reprint,book,0.0
205222,332244,এপারেল মার্চেন্ডাইজিং,হাসান আহমেদ ইমরান,Writer House,Writer House,"Branding, Marketing and Selling","Branding, Marketing and Selling",1st Edition,9789843531209,এপারেল মার্চেন্ডাইজিং বইটি পোশাক শিল্পের মার্চ...,...,5.0,0,0,1190.0,1107.0,https://www.rokomari.com/book/332244/apparel-m...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0
205353,290738,ব্যাঙ কীভাবে বিল-ঝিলের রাজা হলো,কাজল শরিফ,Vorer Shishier,Vorer Shishier,When 4-8: Fables,When 4-8: Fables,2nd edition,978-984-93510-4-7,গল্পটি বাচ্চাদের জন্য বেশ মজার একটি রূপ কথার গ...,...,5.0,0,0,130.0,112.0,https://www.rokomari.com/book/290738/how-a-fro...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0
205385,202072,কবিতা এক দুই তিন চার,মম রহমান,Creative Dhaka Limited,Creative Dhaka Limited,Bangla-English Poem,Bangla-English Poem,1st Published,9789848071151,No summary,...,5.0,0,0,400.0,344.0,https://www.rokomari.com/book/202072/poems-ek-...,https://img.cf.rokomari.com/ProductNew20190903...,Add to Cart,book,0.0


In [21]:
# only keep the bangla characters and discard others
df_bn['bangla_title'] = df_bn['title'].apply(keep_only_bangla)

In [24]:
# sanity check if everthing got erased or subbed with space
df_bn.loc[(df_bn['bangla_title'] == ' ') | (df_bn['bangla_title'] == '')]

,book_id,title,author,publisher,publisher_name_english,categories,category_english,edition,isbn,summary,...,n_ratings,n_reviews,price,offer_price,book_url,prod_img_link,availability,product_category,actual_rating,bangla_title


In [33]:
# save the dataframe

df_bn.to_csv("rokomari_books_only_bangla_v2.csv", index=False)